# 10 - SNOMED CT to US ICD-10-CM codes, free and without a licence application (run_02)

**Procedure (for the record):**
1. **Primary source:** the official NLM *SNOMED CT to ICD-10-CM* map (the same map NLM ships with the SNOMED CT US Edition),
   as republished by the Tuva Project as a public file. Direct download, no account, no API key:
   `https://tuva-public-resources.s3.amazonaws.com/versioned_terminology/latest/snomed_icd_10_map.csv_0_0_0.csv.gz`
2. **Rule used to pick one code per SNOMED concept:** map group 1, rules `TRUE` or `OTHERWISE TRUE` only
   (the NLM default when nothing else is known about the patient), lowest map priority. Targets that are empty or
   not a valid ICD-10-CM code (for example "no target" advice rows) are dropped.
3. **Manual review for the concepts the map does not cover (25 in run_02):** the automatic text search (NLM Clinical
   Tables, free, no key) picked wrong codes (late-effect "sequela" codes, birth injuries, family history), so these are
   assigned by hand from the ICD-10-CM index, using the "initial encounter" (A) 7th character because Synthea records
   the acute event. Flagged `source = manual`. Text search is only used for anything still left and flagged `text_search`.
4. **Left as SNOMED:** concepts with no map row and no search hit (mostly findings and social history,
   which have no ICD-10-CM equivalent).

Mapped: `conditions.csv` CODE and `encounters.csv` REASONCODE. Output: `icd10cm_map.csv` and `conditions_icd10cm.parquet`.
Procedures: there is no free official SNOMED to ICD-10-PCS map, so procedure codes stay SNOMED.

Runtime: CPU, about 2-5 minutes.

In [ ]:
# Cell 1 - download the free map and load the distinct Synthea codes
import os, sys, re, io, gzip, json, time, urllib.request, urllib.parse, pandas as pd
if "google.colab" in sys.modules:
    from google.colab import drive; drive.mount('/content/drive')
    BASE = "/content/drive/MyDrive/genai_synthea/run_02"
else:
    BASE = os.environ.get("RUN_BASE", "run_02")
RAW, OUT = f"{BASE}/raw/csv", f"{BASE}/icd10cm"
os.makedirs(OUT, exist_ok=True)
URL = "https://tuva-public-resources.s3.amazonaws.com/versioned_terminology/latest/snomed_icd_10_map.csv_0_0_0.csv.gz"
GZ = f"{OUT}/snomed_icd_10_map.csv.gz"
if not os.path.exists(GZ): urllib.request.urlretrieve(URL, GZ)
raw = pd.read_csv(GZ, header=None, dtype=str, low_memory=False)
if not raw.iloc[0].str.fullmatch(r"\d+").any():               # first row is a header
    raw.columns = raw.iloc[0].str.lower().str.replace(r"[^a-z]", "", regex=True); raw = raw.iloc[1:]
print(f"Map rows {len(raw):,}, columns {raw.shape[1]}"); print(raw.head(3).to_string())

cond = pd.concat([c for c in pd.read_csv(f"{RAW}/conditions.csv", usecols=["CODE", "DESCRIPTION"], dtype=str, chunksize=500_000)])
enc = pd.read_csv(f"{RAW}/encounters.csv", usecols=["REASONCODE", "REASONDESCRIPTION"], dtype=str).dropna()
codes = pd.concat([cond.rename(columns={"CODE": "snomed", "DESCRIPTION": "desc"}),
                   enc.rename(columns={"REASONCODE": "snomed", "REASONDESCRIPTION": "desc"})]).drop_duplicates("snomed")
print(f"Distinct SNOMED codes in Synthea (conditions + admission reasons): {len(codes):,}")

In [ ]:
# Cell 2 - find the map columns (works with or without a header), apply the NLM default rule
ICD = re.compile(r"^[A-TV-Z][0-9][0-9A-Z](\.?[0-9A-Z]{1,4})?$")
def find(pred):
    s = {c: pred(raw[c].dropna().head(20000)) for c in raw.columns}; return max(s, key=s.get)
want = set(codes.snomed)
c_src = find(lambda v: v.isin(want).mean())                                   # SNOMED concept id
c_tgt = find(lambda v: v.str.match(ICD).mean())                               # ICD-10-CM target
c_rule = find(lambda v: v.str.contains("TRUE", na=False).mean())              # map rule
named = {k: [c for c in raw.columns if k in str(c)] for k in ("group", "priority")}
if named["group"] and named["priority"]: c_grp, c_pri = named["group"][0], named["priority"][0]
else:                                                                         # SNOMED layout: ..., mapGroup, mapPriority, mapRule, ...
    i = list(raw.columns).index(c_rule); c_grp, c_pri = raw.columns[i - 2], raw.columns[i - 1]
print("Detected columns -> source:", c_src, "| target:", c_tgt, "| rule:", c_rule, "| group:", c_grp, "| priority:", c_pri)

m = raw[raw[c_src].isin(want)].copy()
m["icd"] = m[c_tgt].str.replace(".", "", regex=False).str.strip()
m = m[m.icd.str.match(ICD, na=False)]
m = m[m[c_rule].str.upper().str.strip().isin(["TRUE", "OTHERWISE TRUE"])]
if c_grp: m = m[m[c_grp].astype(int) == 1]
if c_pri: m = m.sort_values(c_pri, key=lambda s: s.astype(int))
best = m.drop_duplicates(c_src)[[c_src, "icd"]].rename(columns={c_src: "snomed"}).assign(source="nlm_map")
print(f"Mapped by the official map: {len(best):,} / {len(codes):,}")

In [ ]:
# Cell 3 - fallback: free NLM Clinical Tables text search for codes the map does not cover (flagged for review)
API = "https://clinicaltables.nlm.nih.gov/api/icd10cm/v3/search?sf=code,name&maxList=1&terms="
def search(desc):
    q = re.sub(r"\s*\((disorder|finding|situation|event|morphologic abnormality)\)", "", desc)
    try:
        r = json.load(urllib.request.urlopen(API + urllib.parse.quote(q), timeout=20))
        return r[3][0][0] if r[0] else None
    except Exception: return None
# Hand-checked ICD-10-CM codes for concepts the official map does not cover (initial encounter for injuries)
MANUAL = {
 "Sprain (morphologic abnormality)": "T148XXA", "Sprain of ankle (disorder)": "S93409A", "Normal pregnancy (finding)": "Z3490",
 "Fracture of bone (disorder)": "T148XXA", "Concussion with loss of consciousness (disorder)": "S060X9A",
 "Sprain of wrist (disorder)": "S63509A", "Laceration of thigh (disorder)": "S71119A", "Laceration of foot (disorder)": "S91319A",
 "Injury of tendon of the rotator cuff of shoulder (disorder)": "S46009A", "Laceration of forearm (disorder)": "S51819A",
 "Overdose (disorder)": "T50901A", "Malignant neoplasm of breast (disorder)": "C50919", "Burn injury (morphologic abnormality)": "T300",
 "Fracture of forearm (disorder)": "S5290XA", "Fracture of clavicle (disorder)": "S42009A", "Laceration of hand (disorder)": "S61419A",
 "Injury of neck (disorder)": "S199XXA", "Fracture of rib (disorder)": "S2239XA", "Injury of knee (disorder)": "S8990XA",
 "Fracture of mandible (disorder)": "S02609A", "Traumatic brain injury (disorder)": "S069X9A", "Injury of heart (disorder)": "S2690XA",
 "Tear of meniscus of knee (disorder)": "S83209A", "Injury of kidney (disorder)": "S37009A", "Sudden cardiac death (disorder)": "I469"}
todo = codes[~codes.snomed.isin(best.snomed)]
man = todo[todo.desc.isin(MANUAL)].assign(icd=lambda d: d.desc.map(MANUAL), source="manual")[["snomed", "icd", "source"]]
todo = todo[~todo.desc.isin(MANUAL)]
hits = []
for s, d in zip(todo.snomed, todo.desc):
    hits.append({"snomed": s, "icd": search(d), "source": "text_search"}); time.sleep(0.1)
fb = pd.DataFrame(hits, columns=["snomed", "icd", "source"])
fb["icd"] = fb.icd.str.replace(".", "", regex=False)
fb.loc[fb.icd.isna(), "source"] = "none (kept SNOMED)"
MAP = codes.merge(pd.concat([best, man, fb]), on="snomed", how="left")
MAP.to_csv(f"{OUT}/icd10cm_map.csv", index=False)
print(MAP.source.value_counts().to_string())
print("\nText-search matches to review:"); print(MAP[MAP.source == "text_search"][["desc", "icd"]].to_string(index=False))
print("\nLeft as SNOMED (no ICD-10-CM equivalent expected):"); print(MAP[MAP.source.str.startswith("none")].desc.to_string(index=False))

In [ ]:
# Cell 4 - coverage on actual rows (not just distinct codes) and write the mapped conditions table
C = pd.concat([c for c in pd.read_csv(f"{RAW}/conditions.csv", dtype=str, chunksize=500_000)])
C = C.merge(MAP[["snomed", "icd", "source"]].rename(columns={"snomed": "CODE", "icd": "ICD10CM", "source": "ICD_SOURCE"}), on="CODE", how="left")
C.to_parquet(f"{OUT}/conditions_icd10cm.parquet", index=False)
print("Share of condition ROWS by mapping source:"); print(C.ICD_SOURCE.value_counts(normalize=True).round(3).to_string())
E = pd.read_csv(f"{RAW}/encounters.csv", usecols=["ENCOUNTERCLASS", "REASONCODE"], dtype=str)
E = E[(E.ENCOUNTERCLASS == "inpatient") & E.REASONCODE.notna()].merge(MAP.rename(columns={"snomed": "REASONCODE"}), on="REASONCODE", how="left")
print("\nInpatient admission reasons by mapping source:"); print(E.source.value_counts(normalize=True).round(3).to_string())